# PyTorch 기반 최근접 이웃(KNN) 분류 실습 with 차원축소

## 실습 목표

최근접 이웃(k-Nearest Neighbors, KNN) 분류 흐름을 PyTorch 코드로 구현한 것입니다.

핵심 흐름은 다음과 같습니다.

1. Wisconsin 유방암 데이터 불러오기
2. `id` 컬럼 제거
3. `diagnosis` 라벨 변환
4. StandardScaler 표준화
5. 훈련/테스트 데이터 분리
6. KNN 분류 예측
    - PyTorch는 KNN 전용 모델을 기본 제공하지 않기 때문에, 여기서는 `torch.cdist()`를 사용하여 거리 계산 기반 KNN 알고리즘을 직접 구현합니다.
7. 혼동행렬과 정확도 평가
8. 여러 `k` 값 비교
9. 차원 축소 알고리즘 적용
```text
→ PCA 적용
→ LDA 적용
→ UMAP 적용
→ Autoencoder 적용
→ 성능 비교
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import umap

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("라이브러리 준비 완료")
print("사용 장치:", device)

In [ ]:
breast = load_breast_cancer()

X_df = pd.DataFrame(breast.data, columns=breast.feature_names)
y_series = pd.Series(breast.target, name="diagnosis_raw")

print("sklearn target_names:", breast.target_names)

y_label = y_series.map({1: "Benignant", 0: "Malevolent"})
y_encoded = y_label.map({"Benignant": 0, "Malevolent": 1}).values

print("입력 데이터 크기:", X_df.shape)
print("라벨 데이터 크기:", y_encoded.shape)

display(X_df.head())

print("라벨 분포:")
print(y_label.value_counts())

print("라벨 비율(%):")
print(round(y_label.value_counts(normalize=True) * 100, 2))

In [ ]:
selected_columns = ["mean radius", "mean area", "mean texture"]

print("주요 특징 요약 통계:")
display(X_df[selected_columns].describe())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_df, y_encoded, test_size=0.2, random_state=SEED, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled:", X_train_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
X_train_standard_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
X_test_standard_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)

y_train_tensor = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long).to(device)

print("X_train_standard_tensor:", X_train_standard_tensor.shape)
print("X_test_standard_tensor:", X_test_standard_tensor.shape)
print("y_train_tensor:", y_train_tensor.shape)
print("y_test_tensor:", y_test_tensor.shape)

In [ ]:
class TorchKNNClassifier:
    def __init__(self, k=23):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X_train, y_train):
        self.X_train = X_train
        self.y_train = y_train
        return self

    def predict(self, X_test):
        if self.X_train is None or self.y_train is None:
            raise ValueError("먼저 fit(X_train, y_train)을 호출해야 합니다.")

        distances = torch.cdist(X_test, self.X_train, p=2)
        nearest_indices = torch.topk(distances, k=self.k, dim=1, largest=False).indices
        nearest_labels = self.y_train[nearest_indices]

        predictions = []
        for labels in nearest_labels:
            vote_counts = torch.bincount(labels, minlength=2)
            pred = torch.argmax(vote_counts)
            predictions.append(pred)

        predictions = torch.stack(predictions)
        return predictions

    def score(self, X_test, y_test):
        y_pred = self.predict(X_test)
        correct = (y_pred == y_test).float()
        accuracy = correct.mean().item()
        return accuracy

In [ ]:
knn_model = TorchKNNClassifier(k=23)

knn_model.fit(X_train_standard_tensor, y_train_tensor)

y_pred_tensor = knn_model.predict(X_test_standard_tensor)

accuracy = knn_model.score(X_test_standard_tensor, y_test_tensor)

y_pred = y_pred_tensor.cpu().numpy()
y_true = y_test_tensor.cpu().numpy()

print(f"Standard 정규화 + K=23 정확도: {accuracy:.4f}")

In [ ]:
class_names = ["Benignant", "Malevolent"]

cm = confusion_matrix(y_true, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual_" + name for name in class_names],
    columns=["Pred_" + name for name in class_names]
)

print("혼동행렬:")
display(cm_df)

print("분류 리포트:")
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
k_values = [1, 5, 11, 15, 21, 23, 27]
results = []

for k in k_values:
    model = TorchKNNClassifier(k=k)

    model.fit(X_train_standard_tensor, y_train_tensor)

    pred_tensor = model.predict(X_test_standard_tensor)

    acc = model.score(X_test_standard_tensor, y_test_tensor)

    pred = pred_tensor.cpu().numpy()

    cm_current = confusion_matrix(y_true, pred)

    results.append({
        "k": k,
        "accuracy": acc,
        "confusion_matrix": cm_current
    })

    print(f"k={k:2d}, accuracy={acc:.4f}")

In [ ]:
results_df = pd.DataFrame([
    {"k": item["k"], "accuracy": item["accuracy"]}
    for item in results
])

print("k 값별 정확도:")
display(results_df.sort_values(by="accuracy", ascending=False))

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(results_df["k"], results_df["accuracy"], marker="o")
plt.xlabel("k value")
plt.ylabel("Accuracy")
plt.title("KNN Accuracy by k Value")
plt.xticks(k_values)
plt.grid(True)
plt.show()

# 축소 알고리즘

## 적용 알고리즘

1. PCA
2. LDA
3. t-SNE
4. UMAP
5. Autoencoder

## PCA

In [ ]:
pca_bin = PCA(n_components=2)
X_train_pca = pca_bin.fit_transform(X_train_scaled)
X_test_pca = pca_bin.transform(X_test_scaled)

X_train_pca_tensor = torch.tensor(X_train_pca, dtype=torch.float32).to(device)
X_test_pca_tensor = torch.tensor(X_test_pca, dtype=torch.float32).to(device)

knn_pca_model = TorchKNNClassifier(k=5)
knn_pca_model.fit(X_train_pca_tensor, y_train_tensor)

y_pred_pca_tensor = knn_pca_model.predict(X_test_pca_tensor)
acc_bin_pca = knn_pca_model.score(X_test_pca_tensor, y_test_tensor)

y_pred_pca = y_pred_pca_tensor.cpu().numpy()
y_true = y_test_tensor.cpu().numpy()

print(f"PCA 2D 정확도: {acc_bin_pca:.4f}")
print(f"PCA 2D 누적 설명 분산 비율: {pca_bin.explained_variance_ratio_.sum():.4f}")
print(classification_report(y_true, y_pred_pca, target_names=class_names))

## LDA

In [ ]:
lda_1d = LinearDiscriminantAnalysis(n_components=1)
X_train_lda = lda_1d.fit_transform(X_train_scaled, y_train)
X_test_lda = lda_1d.transform(X_test_scaled)

X_train_lda_tensor = torch.tensor(X_train_lda, dtype=torch.float32).to(device)
X_test_lda_tensor = torch.tensor(X_test_lda, dtype=torch.float32).to(device)

knn_lda_model = TorchKNNClassifier(k=5)
knn_lda_model.fit(X_train_lda_tensor, y_train_tensor)

y_pred_lda_tensor = knn_lda_model.predict(X_test_lda_tensor)
acc_bin_lda = knn_lda_model.score(X_test_lda_tensor, y_test_tensor)

y_pred_lda = y_pred_lda_tensor.cpu().numpy()
y_true = y_test_tensor.cpu().numpy()

print(f"LDA 1D 정확도: {acc_bin_lda:.4f}")
print(f"LDA 1D 누적 설명 분산 비율: {lda_1d.explained_variance_ratio_.sum():.4f}")
print(classification_report(y_true, y_pred_lda, target_names=class_names))

## UMAP

In [ ]:
umap_model = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=SEED)
X_train_umap = umap_model.fit_transform(X_train_scaled)
X_test_umap = umap_model.transform(X_test_scaled)

X_train_umap_tensor = torch.tensor(X_train_umap, dtype=torch.float32).to(device)
X_test_umap_tensor = torch.tensor(X_test_umap, dtype=torch.float32).to(device)

knn_umap_model = TorchKNNClassifier(k=5)
knn_umap_model.fit(X_train_umap_tensor, y_train_tensor)

y_pred_umap_tensor = knn_umap_model.predict(X_test_umap_tensor)
acc_umap = knn_umap_model.score(X_test_umap_tensor, y_test_tensor)

y_pred_umap = y_pred_umap_tensor.cpu().numpy()
y_true = y_test_tensor.cpu().numpy()

print(f"UMAP 2D 정확도: {acc_umap:.4f}")
print(classification_report(y_true, y_pred_umap, target_names=class_names))

## Autoencoder

In [ ]:
ae_latent_dim_list = [2, 3, 4]
ae_practice_results = []
input_dim = X_train_scaled.shape[1]

class PracticeAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(PracticeAutoencoder, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, latent_dim)
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, input_dim)
        )
    
    def forward(self, x):
        z = self.encoder(x)
        reconstructed = self.decoder(z)
        return reconstructed

X_train_tensor_practice = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
X_test_tensor_practice = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)

for latent_dim in ae_latent_dim_list:
    ae_model = PracticeAutoencoder(input_dim=input_dim, latent_dim=latent_dim).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(ae_model.parameters(), lr=0.001)
    epochs = 100
    
    for epoch in range(epochs):
        ae_model.train()
        optimizer.zero_grad()
        reconstructed = ae_model(X_train_tensor_practice)
        loss = criterion(reconstructed, X_train_tensor_practice)
        loss.backward()
        optimizer.step()
    
    ae_model.eval()
    with torch.no_grad():
        X_train_ae = ae_model.encoder(X_train_tensor_practice)
        X_test_ae = ae_model.encoder(X_test_tensor_practice)
    
    X_train_ae_tensor = X_train_ae
    X_test_ae_tensor = X_test_ae

    knn_ae_model = TorchKNNClassifier(k=5)
    knn_ae_model.fit(X_train_ae_tensor, y_train_tensor)
    
    y_pred_ae_tensor = knn_ae_model.predict(X_test_ae_tensor)
    acc_ae = knn_ae_model.score(X_test_ae_tensor, y_test_tensor)
    
    y_pred_ae = y_pred_ae_tensor.cpu().numpy()
    y_true = y_test_tensor.cpu().numpy()
    
    print(f"AE {latent_dim}D + KNN(5) 정확도: {acc_ae:.4f}")
    print(classification_report(y_true, y_pred_ae, target_names=class_names))
    
    ae_practice_results.append((latent_dim, acc_ae))

## 성능비교

In [ ]:
best_ae_acc = max(ae_practice_results, key=lambda x: x[1])

total_results_df = pd.DataFrame({
    "method": [
        "Original KNN(5)",
        "PCA 2D + KNN(5)",
        "LDA 1D + KNN(5)",
        "UMAP + KNN(5)",
        f"AE Best Latent({best_ae_acc[0]}) + KNN(5)"
    ],
    "accuracy": [
        results_df['accuracy'].max(),
        acc_bin_pca,
        acc_bin_lda,
        acc_umap,
        best_ae_acc[1]
    ]
})

total_results_df.sort_values(by="accuracy", ascending=False).reset_index(drop=True)